### [1] Setup & Import

git add . \
git commit -m "Chú thích việc vừa làm" \
git push (Không cần -u origin master nữa, chỉ cần git push là nó tự bay lên đúng chỗ).\

In [ ]:
from contextlib import redirect_stdout
from datetime import datetime
# from scipy.interpolate import pchip_interpolate
import plotly.express as px
import pandas as pd
import numpy as np
import duckdb
import hashlib
import base64
import json
import sys
import io
import re
import os
from dotenv import load_dotenv

if True: #! Load .env | .json
    load_dotenv(dotenv_path='../config/SECRET.env', override=True) 
    cust_salt = int(os.getenv('CUST_SALT_KEY'))
    prod_salt = int(os.getenv('PRODUCT_SALT_KEY'))
    ran_seed  = int(os.getenv('RANDOM_SEED'))
    traffic_scale = [float(x) for x in os.getenv('TRAFFIC_SCALE').split(',')]

with open('../config/DYNAMIC_PRICE.json', 'r') as js:
    scale_dict = json.load(js)

if True: # src setups
    sys.path.append(os.path.abspath(".."))
    %load_ext autoreload
    %autoreload 2

if True: #! Load src
    # 1. is_ -> Bool
    from src.utils import is_boolean, is_datetime, is_alo, is_money, is_numeric, is_category

    # 2. revenue validating
    from src.revenue_logic import cal_revenue, rev_validate

    # 3. date/time validating
    from src.datetime_logic import chunks_maker, validate_n_correct_chunks, recover_date, time_format

    # 4. Pipe
    from src.pipeline_logic import stage_0, stage_1, execution

    #.5 Customer_pipe
    import src.customer_logic as c

    #.6 Product_master
    from src.process_product_master import process_product_master

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Check: 1.12,1.18


[1.12, 1.18]

### [2] Control Center

In [56]:
# 1. Input ----------------------------------------------------------------------------
csv_2024     = r'../CSV_read_only/LMHN_2024_CLEAN.csv'
csv_2025     = r'../CSV_read_only/LMHN_2025_DIRTY.csv'
product_info = r'../CSV_read_only/UPDATE ALL PRICE - JAMES.csv'

config = {
    'payment_cols': ['cash', 'card', 'payoo', 'banking', 'mkt', 'vnpay', 'trade_in'],
    'disc_cols'   : ['disc_percent', 'disc_amount'],
    'date_anchor' : 'invoice',
    'date_pocket' : 'date',
    'drop_col'    : ['vat', 'note'],
    'anonymous'   : ['phone', 'name', 'email',]
    }

f = io.StringIO()

# 2. Load & Normalize -----------------------------------------------------------------
with redirect_stdout(f):
    # - Consolidate disparate datasets into a unified source, standardize header naming
    df_2024 = pd.read_csv(csv_2024)
    df_2025 = pd.read_csv(csv_2025)
    df_2024.columns = df_2025.columns

    first_4_cols = df_2024.columns[:4]
    df_raw = pd.concat([df_2024, df_2025], axis=0, ignore_index=True).dropna(subset=first_4_cols, ignore_index=True)

    df_raw.columns = df_raw.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.lower()
    df_raw = df_raw.drop(config['drop_col'], axis=1, errors='ignore')

    df = df_raw.copy()

# 3. Data Validation & Standardization ------------------------------------------------
with redirect_stdout(f):
    # - Automate column classification using regex and sample-based testing.
    # - Repair corrupted timestamps using anchor-based logic and gap filling.
    # - Reconstruct missing revenue using cross-column financial validation.
    categorize_results = stage_1(stage_0(df))
    df = rev_validate(df, config['payment_cols'], config['disc_cols'], categorize_results)

    df[config['date_pocket']], list_errors_date = recover_date(df, config['date_pocket'], config['date_anchor'])
    df = df.drop('fill_date', axis=1, errors='ignore')

    #* After cleaned
    df_new = execution(df, categorize_results).copy()

# 4. Identity Masking & Privacy Protection --------------------------------------------
with redirect_stdout(f):
    # - Encrypting customer PII (Phone/Name/Email) using Salted Base32.
    cust_cols = config.get('anonymous')
    _p, _n, _e = cust_cols

    # Normalize before create Cust Master
    df_new = c.cus_normalize(df_new, _p, _n, _e)
    # Creating Cust Master
    df_cust_master = c.create_cust_master(df_new, _p, _n, _e)
    df_cust_master = c.base32_encode(df_cust_master, _p, cust_salt)
    df_cust_master = c.create_cus_id(df_cust_master, _p, _n, _e)
    # Encode main df[phone]
    df_new = c.base32_encode(df_new, _p, cust_salt).drop([ _n, _e], axis=1, errors='ignore')

    # Map Cust Master back to main df
    df_new = df_new.merge(df_cust_master, how='left', on=_p)

# 5. Product Master Restructuring & Price Masking -------------------------------------
with redirect_stdout(f):
    # - Parse raw_descriptions to extract structured Color and Memory attributes.
    # - Dynamic price scaling with controlled noise to protect margins.

    # -----------------------------------------
    # product_master = process_product_master(
    # product_info=product_info,
    # prod_salt=prod_salt,
    # ran_seed=ran_seed,
    # scale_dict=scale_dict,
    # output_path='../CSV_write/Anonym_Price.csv'
    # )
    pass

# 6. Applying hashed SKU, product fingerprints hashing --------------------------------
with redirect_stdout(f):
    # - Generate cryptographic fingerprints for product IMEI/Serials via SHA-256.

    # Result from step 5.
    anonym_price = pd.read_csv('../CSV_write/Anonym_Price.csv')
    # Normalize dtype and format in order to mapping
    def normalize4_lookup(series:pd.Series)-> pd.Series:
        """
        Convert dtype & format of lookup key.\n
        Remove '.0' at the end of EAN string.\n
        Strip.
        """
        return series.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    anonym_price['ean'] = anonym_price['ean'].pipe(normalize4_lookup)
    df_new['ean']       = df_new['ean'].pipe(normalize4_lookup)

    # Create lookup_Series with index=key_word
    lookup_dict = anonym_price.set_index('ean')['master_sku']
    # Create 'sku' column from Anonym_Price
    df_new.insert(4, 'sku', df_new['ean'].map(lookup_dict))

    def imei_hashing(series: pd.Series, key) -> pd.Series:
        unique_imeis = series[series!='unknown'].dropna().unique()
        
        salt = str(key).encode()
        hash_map = {}
        for imei in unique_imeis:
            h = hashlib.sha256(str(imei).encode() + salt).digest()
            hash_map[imei] = base64.b32encode(h).decode()[0:50:5]
    
        return series.map(hash_map).fillna('unknown')
    
    df_new['imei_sn'] = imei_hashing(df_new['imei_sn'], cust_salt)

# 7. Applying anonymized product infos -> df_ready ------------------------------------
with redirect_stdout(f):
    # - Inject anonymized product features

    # Preparing Main df
    sensitive_cols = [ 'sap_article', 'sap_description'] #! 'ean',
    df_new = df_new.drop(sensitive_cols, axis=1, errors='ignore')

    # Preparing price df
    anonym_price['detail_sub_lob'] = anonym_price['detail_sub_lob'].astype('string').str.strip().str.title()
    anonym_price = anonym_price.rename({'master_sku': 'sku'}, axis=1)
    anonym_cols = ['sku', 'product_name', 
        'detail_sub_lob', 'color', 
        'memory_size', 'new_price']
    
    # Merging & Re-ordering main DataFrame
    df_ready = df_new.merge(anonym_price[anonym_cols], how='left', on='sku')
    _orders = ['date', 'invoice', 
        'sa', 'sku', 'imei_sn', 
        'cat', 'detail_sub_lob', 
        'product_name', 'new_price']
    ready_orders = _orders + [col for col in df_ready.columns if col not in _orders]
    df_ready = df_ready[ready_orders]

# 8. Pricing history mimicking & Restructuring payment methods ------------------------
with redirect_stdout(f):
    # - Extracting originals product prices from revenue, filling missing price.
    # - Extracting pricing ratios per SKU throughout history.
    # - Re applying ratios into anonymized price (preserving data integrity)
    # - Compute payment ratios then grouping into 3 groups

    def fill_origin_price(df: pd.DataFrame, col_map: dict = None, price_ratio: bool = False
    ) -> pd.DataFrame:
        """
        #### Tính toán giá gốc dựa trên Revenue, Price, Qty, Discount methods.
        #### Tạo price_ratio cho từng row (price_ratio=True)
        default_cols if col_map = None:
            revenue, disc_amount, disc_percent, qty, price, ean
        """

        default_cols = {
            'revenue': 'revenue',
            'disc_amount': 'disc_amount',
            'disc_percent': 'disc_percent',
            'qty': 'qty',
            'price': 'price',
            'ean': 'ean'
        }

        cols = default_cols | (col_map or {})

        df = df.copy()

        # calculate origin price
        origin = (
            (df[cols['revenue']] + df[cols['disc_amount']]) /
            ((1 - df[cols['disc_percent']]) * df[cols['qty']])
        ).fillna(df[cols['price']])

        # Create price_dict ( original price for each EAN )
        price_dict = (
            origin
            # Thao tác trên Series không set_index() được nên dùng set_axis() hoặc s.index() nếu không cần chain
            .set_axis(df[cols['ean']])
            .groupby(level=0)
            .first()
            .replace([np.inf, -np.inf], np.nan)
            .rename(cols['price'])
            .reset_index()
            .dropna()
        )
        # Astype for lookup
        price_dict[cols['ean']] = (pd.to_numeric(price_dict[cols['ean']], errors='coerce').astype('Int64').astype('string'))

        # Groupby 1 lần nữa sau khi đồng nhất ean
        price_dict = price_dict.groupby(cols['ean']).first() 
        price_dict = price_dict[cols['price']]

        #* Re-creating price col
        raw_price_backup = df[cols['price']].copy().replace([np.inf, -np.inf], np.nan)
        df[cols['price']] = df[cols['ean']].map(price_dict).fillna(raw_price_backup)
        print(f'Debug [fill_origin_price] invalid Re-creating price rows: {df[cols['price']].isna().sum()}')

        if price_ratio:
            first_price = df.groupby(cols['ean'])[cols['price']].transform('first')
            print(f'Debug [fill_origin_price] (1st_price == 0 | missing) count: {((first_price == 0) | (first_price.isna())).sum()}') 
            df['price_ratio'] = first_price / df[cols['price']]

        return df
    df_ready = fill_origin_price(df_ready, price_ratio=True)

    # --------------- Apply price trend/ Replace original price col ---------------
    if 'new_price' in df_ready.columns:  
        df_ready['test_price'] = df_ready['new_price'].astype(float)
        p_ratio_mask = (df_ready['price_ratio'] != 1) & df_ready['price_ratio'].notna()
        df_ready.loc[p_ratio_mask, 'test_price'] = (
            np.floor(
                (df_ready.loc[p_ratio_mask, 'test_price'] * df_ready.loc[p_ratio_mask, 'price_ratio']) / 200_000
                ) * 200_000 + 90_000
            )
        df_ready['price'] = df_ready['test_price']
    to_clean = ['new_price', 'phone', 'price_ratio', 'test_price']
    df_ready = df_ready.drop(to_clean, axis=1, errors='ignore')
    # ----- Compute payment ratio for 7 methods then grouping into 3 methods ------
    if 'vnpay' in df_ready.columns:
        raw_payment_ratios = (
            df_ready[config['payment_cols']]
            .div(df_ready['revenue'].mask(df_ready['revenue'] == 0), axis=0)
            .fillna(0)
        )
        # Create compact payment_ratios ['cash', 'card', 'qr_code']
        payment_ratios = pd.DataFrame(0, columns=['cash'], dtype=np.float64, index=df_ready.index)
        # Grouping & assign payment methods
        payment_ratios['cash'] = raw_payment_ratios['cash']
        payment_ratios = payment_ratios.assign(
            card = raw_payment_ratios[['card', 'payoo', 'banking']].sum(axis=1),
            qr_code = raw_payment_ratios[['mkt', 'vnpay', 'trade_in']].sum(axis=1)
        )

        payment_ratios['count'] = (payment_ratios[['cash', 'card', 'qr_code']] != 0).sum(axis=1)


        # Create mask for rounding invalid ratios
        single = payment_ratios['count'] == 1
        valid_1 = np.isclose(payment_ratios[['cash', 'card', 'qr_code']].sum(axis=1), 1)
        valid_0 = np.isclose(payment_ratios[['cash', 'card', 'qr_code']].sum(axis=1), 0)
        round_mask = single | valid_0 | valid_1

        print(f'DEBUG Final invalid payment ratio: {(~round_mask).sum()}')
        payment_ratios.loc[round_mask] = payment_ratios.loc[round_mask].round()
        # Create no_payment mask
        payment_ratios['no_payment'] = (payment_ratios[['cash', 'card', 'qr_code']] != 0).sum(axis=1) < 1
        df_ready = df_ready.drop(config['payment_cols'], axis=1, errors='ignore')

    #------------------------------------------------------------------------------

    # Drop 'revenue' to trigger recomputation in rev_validate()
    # This ensures consistency between re-created prices and revenue.
    if 'revenue' not in df_ready.columns: print("8. 'revenue' does not exist")
    else:
        df_ready = df_ready.drop('revenue', axis=1, errors='ignore')
        print('# 8. ----- Remove original revenue --')

            # stage_0 catagorizing columns results needed
        _, cat_results, _ = stage_0(df_ready)

            # Revenue recomputating
        print('# 8. ----- Re_computing revenue -----')
        df_ready = rev_validate(
            df_ready, 
            disc_cols=config['disc_cols'], 
            results=cat_results)
        print('# 8. ----- Added revenue col --------')

            # Re-creating compact version of payment methods 
        df_ready = df_ready.assign(
            cash = df_ready['revenue'] * payment_ratios['cash'],
            card = df_ready['revenue'] * payment_ratios['card'],
            qr_code = df_ready['revenue'] * payment_ratios['qr_code'],
            no_payment = payment_ratios['no_payment']
        )

            # Mask for final step revenue validation
        payment_error_mask = (
            ~np.isclose(df_ready[['cash', 'card', 'qr_code']].sum(axis=1), df_ready['revenue']) #  revenue != payment.sum()
            | (df_ready['no_payment'])                                                          #  payment_method == 0
        ) & (df_ready['qty'] == 0)                                                              #  qty == 0

            # Fill error rows with NaN
        df_ready.loc[payment_error_mask, ['qty', 'revenue']] = np.nan






# df_ready.insert(1, 'day', df_ready['date'].dt.day)
# df_ready.insert(2, 'month', df_ready['date'].dt.month)
# df_ready.insert(3, 'year', df_ready['date'].dt.year)

df_ready

,date,invoice,sa,sku,imei_sn,cat,detail_sub_lob,product_name,ean,price,qty,ins_stt,ins_fee,disc_percent,disc_amount,time,id,name,email,color,memory_size,revenue,cash,card,qr_code,no_payment
0,2024-04-01,11176,Tom,3RD-CVZFO-M77LA,unknown,3RD ACC,Powerbank,DISPLAY 15W WIRELESS POWER BANK 10000,4895229137837,1529000.0,1.0,False,0.0,0.0,0.0,1900-01-01 22:35:00,NaN,NaN,NaN,NaN,NaN,1529000.0,0.0,1529000.0,0.0,False
1,2024-04-01,11177,Tom,APP-ARM4X-CPM7Q,unknown,ACCESSORIES (APPLE),Charger,20W USB-C POWER ADAPTER,195949121302,590000.0,1.0,False,0.0,0.0,0.0,NaT,NaN,NaN,NaN,NaN,NaN,590000.0,0.0,0.0,590000.0,False
2,2024-04-01,11178,Tom,APP-ARMDO-UGTLQ,unknown,ACCESSORIES (APPLE),Charger,USB-C TO APPLE PENCIL ADAPTER,194253686958,390000.0,1.0,False,0.0,0.0,0.0,NaT,NaN,NaN,NaN,NaN,NaN,390000.0,390000.0,0.0,0.0,False
3,2024-04-01,11179,Charlie,3RD-CVZFO-MPQMQ,unknown,3RD ACC,Cable,MFI SYNC CHARGE C TO LIGHTNING CAB 10M,4895229104112,529000.0,1.0,False,0.0,0.0,0.0,NaT,NaN,NaN,NaN,NaN,NaN,529000.0,0.0,0.0,529000.0,False
4,2024-04-01,11180,Charlie,3RD-DTUAZ-34ZSQ,unknown,3RD ACC,Normal,MIẾNG DÁN CƯỜNG LỰC MIPOW KINGBULL HD PREMIUM-...,6945764522556,319000.0,1.0,False,0.0,0.0,0.0,NaT,NaN,NaN,NaN,NaN,NaN,319000.0,0.0,319000.0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17756,2025-12-31,22361,Lee,APP-ARM4X-2AYGQ,unknown,ACCESSORIES (APPLE),Iphone 17 Pro Max Case,IPHONE 17 PRO MAX CASE CLEAR,195950664164,1490000.0,1.0,False,0.0,0.0,0.0,1900-01-01 21:18:00,NaN,NaN,NaN,Clear,NaN,1490000.0,0.0,1490000.0,0.0,False
17757,2025-12-31,22361,Lee,3RD-DTYFM-OWMKQ,unknown,3RD ACC,Front Screen,JCP4538,6954661873388,489000.0,1.0,False,0.0,0.0,0.0,1900-01-01 21:18:00,NaN,NaN,NaN,NaN,NaN,489000.0,0.0,489000.0,0.0,False
17758,2025-12-31,22362,Lee,3RD-DTUAZ-37I3A,unknown,3RD ACC,Case,CASE ORANGE,6945764527629,599000.0,1.0,False,0.0,0.0,0.0,1900-01-01 17:29:00,NaN,NaN,NaN,Orange,NaN,599000.0,0.0,599000.0,0.0,False
17759,2025-12-31,22362,Lee,3RD-DTUAZ-37ASQ,unknown,3RD ACC,Front Screen,BJ712-BK,6945764527100,489000.0,1.0,False,0.0,0.0,0.0,1900-01-01 17:48:00,NaN,NaN,NaN,NaN,NaN,489000.0,0.0,489000.0,0.0,False


### Extract Price Trend

### Traffic

In [19]:
# Load and consolidate store traffic data (2024-2026) using DuckDB
# Integrating regional holiday data to preserve seasonal correlation
if True: # import Traffic & Holiday files
    store_traffic_2024 = duckdb.sql(
        "SELECT Period::DATE as Period, Traffic     FROM '../CSV_read_only/APPLE_2024_TRAFFIC.csv'")
    store_traffic_2025 = duckdb.sql(
        "SELECT Period::DATE as Period, Traffic     FROM '../CSV_read_only/APPLE_2025_TRAFFIC.csv'")
    store_traffic_2026 = duckdb.sql(
        "SELECT Period::DATE as Period, Traffic     FROM '../CSV_read_only/APPLE_2026_TRAFFIC.csv'")
    holiday_vn = duckdb.sql(
        "SELECT Period::DATE as Period, Event_name  FROM '../CSV_read_only/Holiday_VN_2024_2026.csv'")

total_traffic = duckdb.sql(
"""--sql
WITH traffic AS(
    SELECT * FROM store_traffic_2024
        UNION ALL
    SELECT * FROM store_traffic_2025
        UNION ALL
    SELECT * FROM store_traffic_2026
)
SELECT 
    t.Period, 
    t.Traffic,
    h.Event_name
FROM  traffic t
LEFT JOIN holiday_vn h
ON t.Period = h.Period
ORDER BY t.Period ASC;                                          -- Must be sorted 
""").df()

def nonlinear_rescale(df:pd.DataFrame, metric:str, scale_min_max:list, ran_seed:int) -> pd.DataFrame:
    """
    Synthesizes anonymized metrics while preserving original trends and seasonality.
    Uses Non-linear interpolation combined with static noise for realistic variance.
    """
    if not is_numeric(df[metric]): return
    np.random.seed(ran_seed)

    min_raw = df[metric].nsmallest(2).values[1]
    max_raw = df[metric].nlargest(1).values[0]

    interp_index = np.interp(df[metric], [min_raw, max_raw], scale_min_max)
    noise = pd.Series(np.random.normal(1, 0.31, size=len(df)), index=df.index)
    final_index = interp_index  * np.sin((np.pi /2) ** 0.2) * noise
    processed = df[metric] * final_index

    if f'new_{metric}' in df.columns:
        df.drop(f'new_{metric}', axis=1, inplace=True)
    df.insert(1, f'new_{metric}', processed)

    # Fill low
    low_threshold:float = df[f'new_{metric}'].quantile(0.15)
    is_low = df[f'new_{metric}'] <= low_threshold
    df.loc[is_low, f'new_{metric}'] = np.nan

    df[f'new_{metric}'] = (df[f'new_{metric}']
        .interpolate(method='pchip')
        .round(-1)).astype('Int32')
    if metric in df.columns:
        df.drop(metric, axis=1, inplace=True)
    return df

total_traffic = nonlinear_rescale(total_traffic, 'Traffic', traffic_scale, ran_seed)
total_traffic

,Period,new_Traffic,Event_name
0,2024-01-01,3370,Tết Dương Lịch
1,2024-01-02,440,None
2,2024-01-03,250,None
3,2024-01-04,420,None
4,2024-01-05,310,None
...,...,...,...
833,2026-04-13,510,None
834,2026-04-14,350,None
835,2026-04-15,280,None
836,2026-04-16,230,None


### Analysis_SQL

In [63]:
# ------ config ------
source_df         = 'df_ready'
date_col          = 'date'
time_col          = 'time'
qty_col           = 'qty'
rev_column        = 'revenue'        
event_start_date  = '2025-09-18'    
analysis_end_date = '2025-12-31' 
rolling_window    = 6               
# ----------------------

# astype cols SQL style
duck_new = duckdb.sql(f"""--sql
    SELECT *
    REPLACE (
        {qty_col}::INT32 as qty,
        {date_col}::DATE as date,
        {time_col}::TIMESTAMP as time
    )
    FROM {source_df};
""")
# Phải groupby cho mỗi ngày thành 1 dòng trước 
# vì AVG() nó không quan tâm 1 ngày có bao nhiêu dòng, 
# nếu 1 ngày càng nhiều dòng thì số AVG càng nhỏ và chả phản ánh đúng cái gì cả

duck_date_rev = duckdb.sql(f"""--sql
    WITH pre_cal AS (
        SELECT
            date as Date,
            SUM({rev_column}) as Revenue_by_Date                                            -- #!!!
        FROM duck_new
        GROUP BY date
        ORDER BY date ASC
    ),

    raw_cal AS (
        SELECT
            Date,
            Revenue_by_Date,
            ROUND(AVG(Revenue_by_Date) OVER(
                ORDER BY Date ROWS BETWEEN {rolling_window} PRECEDING AND CURRENT ROW
            ), 0) as Rolling_7Day            
        FROM pre_cal
    ),

    sqrt_abc AS (
        SELECT
            *,
            ROUND(SQRT(Revenue_by_Date), 0) as sqrt_revenue, 
            ROUND(SQRT(Rolling_7Day), 0)    as sqrt_rolling,
            CASE WHEN Date >= '{event_start_date}' AND Date <= '{analysis_end_date}' 
                 THEN ROUND(SQRT(Rolling_7Day), 0) ELSE NULL END as sqrt_event_launching,
            CASE WHEN Date >= '{event_start_date}' AND Date <= '{analysis_end_date}' 
                 THEN 1 ELSE 0 END as is_after_event
        FROM raw_cal
    ),

    mean_abc AS (
        SELECT 
            *,  
            AVG(CASE WHEN is_after_event = 0 THEN Revenue_by_Date ELSE NULL END) OVER() as rbm,
            AVG(CASE WHEN is_after_event = 1 THEN Revenue_by_Date ELSE NULL END) OVER() as ram
        FROM sqrt_abc
    )

    SELECT 
        *,
        CASE WHEN is_after_event = 0 THEN rbm ELSE NULL END as raw_before_event_mean,
        CASE WHEN is_after_event = 1 THEN ram ELSE NULL END as raw_after_event_mean,
        CASE WHEN is_after_event = 0 THEN SQRT(rbm) ELSE NULL END as sqrt_before_event_mean,
        CASE WHEN is_after_event = 1 THEN SQRT(ram) ELSE NULL END as sqrt_after_event_mean,
        (ram / rbm - 1) * 100 AS event_increase
    FROM mean_abc;
""").df()

if True: 
    number_cols = duck_date_rev.select_dtypes(include='number').columns
    duck_date_rev[number_cols] = duck_date_rev[number_cols].round(2).convert_dtypes()

# duck_date_rev

### First Chart
Có thể add vào bất cứ chart nào\
fig.add_scatter( )\
fig.add_bar( )\
fig.add_histogram( )\
fig.add_vline( )\
fig.add_hline( )\
fig.add_annotation( )\
fig.update_traces( )\
fig.update_xaxes( )\
fig.update_yaxes( )\
fig.update_layout( )

In [64]:
tick_vals_raw = [
    0, 200_000_000, 600_000_000, 
    1_200_000_000,  1_800_000_000
    ]
y_max_sqrt = (max(tick_vals_raw) + 4e8) **0.5
tick_text_raw = ['0', '200 Tr', '600 Tr', '1,2 Tỉ', '1,8 Tỉ']
tick_vals_sqrt = np.sqrt(tick_vals_raw).tolist()

if True: #! Params
    #todo Không thể gán value cho vế dùng function/method
    x_raw               = 'Date'
    y_sqrt              = ['sqrt_revenue', 'sqrt_rolling']
    y_raw               = ['Revenue_by_Date', 'Rolling_7Day']
    legend_a            = 'Daily Revenue'
    legend_b            = '7D Trend'

    # title & range
    x_title           = 'Tháng'
    y_title           = 'VNĐ'
    x_range           = ["2025-01-01", "2025-12-31"]
    y_range           = [0, y_max_sqrt]

    # Event & Mean columns
    sqrt_event_end_year = 'sqrt_event_launching' 
    raw_b_mean          = 'raw_before_event_mean'
    raw_a_mean          = 'raw_after_event_mean'
    sqrt_b_mean         = 'sqrt_before_event_mean'
    sqrt_a_mean         = 'sqrt_after_event_mean'
    event_increase      = duck_date_rev.loc[0, 'event_increase']


fig = px.line(duck_date_rev, 
    title = '<b>AAR Store: Doanh thu 2025</b> <span style="font-size:14px; color:#888;">(VNĐ - sqrt scale)</span>',
    x = x_raw, 
    y = y_sqrt,                                       # Truyền list tên cột vào đây (cột để vẽ, có thể đã Square hoặc log)
    custom_data = y_raw, # Custom_data = Values Raw dùng để show kết quả hover. 
    labels = {'variable': '<b>Metric</b>'},            # Để đặt tên trục/ legend
    hover_name = x_raw,
    # color = None,
    color='variable',
    animation_frame=None, #! Test
    render_mode = 'svg'
    # width=1400, height=550
)

# Rolling 7D
fig.update_traces(
    selector={'name': y_sqrt[1]},
    name = legend_b,
    showlegend = True,
    line_color = "#4281AF", 
    line = dict(
        dash = 'solid',
        width = 2.8,
        shape = 'spline',
        smoothing = 1.3
        ),
    opacity = 1,
    fill = 'tozeroy', # Đổ màu từ đường line xuống trục 0
    fillcolor = 'rgba(120, 197, 239, 0.1)', 
    hovertemplate = "<b>Tuần: %{x|%W}</b><br>Trung bình: %{customdata[1]:,.0f} VNĐ<extra></extra>",
    # visible = 'legendonly'
)
# Daily Revenue
fig.update_traces(
    selector={'name': y_sqrt[0]},
    name = legend_a,
    showlegend = True,
    # line_color="#5EAEFF", 
    line_color = "#367050", 
    line = dict(
        width = 0.8, 
        shape = 'hv', 
        smoothing = 1.3
        ),
    opacity = 0.4,
    hovertemplate = "<b>Ngày: %{x|%d-%m-%Y}</b><br>Doanh thu: %{customdata[0]:,.0f} VNĐ<extra></extra>"
    # fill='tozeroy', # Đổ màu từ đường line xuống trục 0
    # fillcolor='rgba(135, 206, 250, 0.18)', # Màu xanh nhạt với độ trong suốt 20%
)
# Event_launch -> End year
fig.add_scatter(
    line_color = "#4281AF",
    x = duck_date_rev[x_raw], 
    y = duck_date_rev[sqrt_event_end_year],
    fill = 'tozeroy',
    # fillcolor='rgba(255, 167, 0, 0.2)',
    fillcolor = 'rgba(120, 200, 40, 0.2)',
    line = dict(width = 2.7,       # Ẩn = 0
                shape = 'spline', 
                smoothing = 1.3),    
    name = '',                      #! Ẩn
    showlegend = False,             # Giấu khỏi bảng chú thích (Legend)
    hoverinfo = 'skip'              # Rê chuột vào không hiện gì cả
)
# Mean line before_npi
fig.add_scatter(
    x=duck_date_rev[x_raw], 
    y=duck_date_rev[sqrt_b_mean], 
    mode='lines', 
    line=dict(color="#467797", width=1.7, dash='longdashdot'), # Chỉnh trực tiếp ở đây là chắc ăn nhất
        # BỘ BA "VÔ HÌNH":
    showlegend=False,        # 1. Giấu khỏi bảng chú thích (Legend)
    hoverinfo='skip',        # 2. Rê chuột vào không hiện gì cả
    name=''                  # 3. Để tên trống để tránh hiện lỗi nếu lỡ hover trúng
)
# Mean line after_npi
fig.add_scatter(
    x=duck_date_rev[x_raw], 
    y=duck_date_rev[sqrt_a_mean], 
    mode='lines', 
    line=dict(color="#4EA073", width=1.7, dash='longdashdot'), # Chỉnh trực tiếp ở đây là chắc ăn nhất
        # BỘ BA "VÔ HÌNH":
    showlegend=False,        # 1. Giấu khỏi bảng chú thích (Legend)
    hoverinfo='skip',        # 2. Rê chuột vào không hiện gì cả
    name=''                  # 3. Để tên trống để tránh hiện lỗi nếu lỡ hover trúng
)


fig.update_yaxes(
    tickmode = "array",
    tickvals = tick_vals_sqrt, # Vị trí đặt vạch chia (trên thang đo sqrt)
    ticktext = tick_text_raw, # Chữ hiển thị tại vạch đó
    tickprefix = None,
    ticksuffix = '    ',
    automargin = True,
    title = y_title,
    range = y_range,
)
fig.update_xaxes(
    ticklabelmode = "period",
    title_text = x_title,
    range = x_range, 
    title_font = dict(size=12),
    dtick = "M1",      # Có thể là 3 tháng, 1 ngày, 1 giờ...
    tickformat = "%b %y", # %b = chuẩn ISO cho tháng dạng Jan,...
    showgrid = True,
    gridwidth = 1, 
    gridcolor = 'white',
    # x_line = off
    showline = False, 
)
# Thêm đường Trần (y = max(y))
fig.add_hline(y = y_max_sqrt, 
              line_dash="solid", 
              line_color="#D3D3D3", 
              line_width=1)
# Thêm đường Sàn (ví dụ 0 VNĐ)
fig.add_hline(y=0, 
              line_dash="solid", 
              line_color="#D3D3D3", 
              line_width=0.5)
# Thêm Event Line
fig.add_vline(
    x = event_start_date, 
    line_width = 3.1, 
    line_dash = "longdash", 
    line_color = "#E8732A",
    layer = 'above'
)
fig.add_annotation(
    x = event_start_date, 
    y = 1.015, yref = "paper",
    text = "<b>🚀 iPhone 17 NPI</b><br>September 19, 2025.",
    align = 'left',
    showarrow = False,
    font = dict(
        family="Segoe UI Semibold",
        size=10,         
        color="#F28136",   
    ),
    xshift = -5,            # Đẩy chữ sang phải 5px để không dính sát vào đường kẻ
    yanchor = "bottom",
    xanchor = "left"
)
fig.add_annotation(
    text=f"<b>▲ {event_increase:.1f}%</b><br>Vượt mức trung bình trước NPI<br>duy trì đến hết năm 2025.",
    x='2025-10-21',  #! x theo df
    y=0.72,          #! y theo paper
    yref="paper",    # Bắt buộc có dòng này để y= có ý nghĩa
    showarrow=False,
    align="left",
    xanchor="left", 
    yanchor="top",  
    font=dict(size=10, color="gray")          
)

#! Theme | Legend | Margin | X-Y show title | Grid_color | Font | Hover_mode.
fig.update_layout(
    template="plotly_white",
    #! Legend
    legend=dict(
        orientation="h",        # Legend nằm ngang
        yanchor="top", y=1.175, 
        xanchor="center", x=0.5,
        font=dict(size=12, color="#86868b")
        ),
    #! Lề
    margin=dict(
        b=40,
        t=80, 
        l=40,  
        r=40),

    #! Font
    font_family='Segoe UI Semibold',
    font_color="#2C528C",
    title_font_size=26,

    #! X-axis
    xaxis=dict(
        title=None,             # Bỏ
        gridcolor="#D3D3D3",
        gridwidth = 0.2,
        showgrid=False,
        tickfont=dict(size=12),
        title_font=dict(size=12)
    ),
    #! Y-axis
    yaxis=dict(
        gridcolor = "#ECECEC",
        gridwidth = 0.1,
        showgrid = False,
        tickfont = dict(size=12),
        title_font = dict(size=12)
    ),
    hovermode="x unified",
    transition_duration=800,
    hoverlabel=dict(bgcolor="rgba(255, 255, 255, 0.9)", font_size=13),

# Thêm Button vào trong fig.update_layout

)
# fig.update_xaxes(rangeslider_visible=True)
fig.show()

#### LOG

In [ ]:
# print(f.getvalue())
# now = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
# with open('../log/cleaning_log.txt', 'a', encoding='utf-8') as logg:
#     logg.write(f"\n--- Run at: {now} ---\n")
#     logg.write(f.getvalue())

# # print(f"[{'Stage_1':<15}] no match col: {'':<20} -> string_col")